# 1. Von Pixeln zu Wörtern — Sentimentanalyse beim Kfz-Versicherer

**Session 8 · Dauer: 30–60 Min**

## Real-World-Kontext

Unser Kfz-Versicherer bekommt nicht nur Fotos von Schäden (das hattest Du schon), sondern vor
allem: **Text**. Jeden Tag laufen hunderte Kundenbewertungen der Versicherer-App ein, dazu
Freitext-Beschreibungen in Schadensmeldungen und E-Mails an den Kundenservice. Kein Mensch kann
das alles einzeln lesen und bewerten — aber genau das wäre nötig, um schnell auf unzufriedene
Kunden zu reagieren oder dringende Schadensfälle zu priorisieren.

Die Lösung: ein Sprachmodell übernimmt die Vorsortierung. Es liest jeden Text automatisch und
schätzt ein, ob die Tonalität eher positiv, neutral oder negativ ist — und wie **sicher** es sich
dabei ist. Genau diese Pipeline (Rohtext → Tokenisierung → Embedding → Modell → Vorhersage)
bauen wir in diesem Notebook Schritt für Schritt nach, mit einem echten, vortrainierten
Transformer-Modell.

> 💡 **Good to know:**
> Wir nutzen dasselbe Grundprinzip wie beim Transfer Learning für Bilder: Statt ein Sprachmodell
> von Grund auf zu trainieren (Wochen an Rechenzeit, Millionen Texte nötig), laden wir ein
> bereits fertig trainiertes Modell von der **HuggingFace**-Plattform und wenden es direkt auf
> unsere eigenen Kfz-Versicherer-Sätze an.

## 🎯 Learning Objectives

By completing this notebook, you will be able to:
- 💡 **Explain** den Unterschied zwischen Word-Level- und Subword-Tokenisierung und warum
  Subword-Tokenisierung mit zusammengesetzten deutschen Wörtern besser umgehen kann.
- 🔍 **Identify** ein Embedding als Vektor im n-dimensionalen Raum und Ähnlichkeit über
  Kosinus-Ähnlichkeit berechnen.
- 🛠️ **Apply** `AutoTokenizer` und `AutoModelForSequenceClassification` aus der
  `transformers`-Bibliothek, um ein vortrainiertes Sentiment-Modell zu laden und auf eigene
  deutsche Sätze anzuwenden.
- 🛠️ **Apply** den vollständigen Inferenz-Ablauf (Tokenisieren → Forward Pass →
  `torch.softmax()` → `argmax()` → `id2label`) sowohl auf einzelne Sätze als auch auf einen
  ganzen Batch von Kundenbewertungen.
- ⚖️ **Bewerten**, wann eine niedrige Modell-Konfidenz ein Signal für manuelle Prüfung ist, und
  die Grenzen von Sentimentanalyse bei Sarkasmus und Negation kritisch einordnen.

## Concept at a Glance

**Analogie — der Tokenizer als Dolmetscher:** Ein Transformer-Modell versteht keine Buchstaben,
sondern ausschließlich **Zahlen**. Stell Dir den **Tokenizer** deshalb wie einen Dolmetscher vor,
der jeden Satz zunächst in kleine, nummerierte Wort- bzw. Wortteil-Bausteine zerlegt — das sind
die **Tokens**. Jeder Baustein wird dann über eine riesige, trainierte Nachschlagetabelle (das
**Embedding**) in einen Vektor übersetzt: eine Liste von Zahlen, die die Bedeutung des Bausteins
codiert. Erst danach kann das eigentliche Transformer-Modell mit **Attention** die Beziehungen
zwischen den Vektoren auswerten und eine Vorhersage treffen.

Die komplette Pipeline, die wir in diesem Notebook nachbauen:

$$\text{Rohtext} \;\rightarrow\; \text{Tokenisierung} \;\rightarrow\; \text{Embedding} \;\rightarrow\; \text{Modell (Attention)} \;\rightarrow\; \text{Vorhersage}$$

> 💡 **Good to know:**
> Attention selbst hast Du bereits konzeptionell kennengelernt: Jedes Wort "schaut" auf jedes
> andere Wort im Satz und gewichtet, wie relevant es dafür ist. Dieses Notebook setzt genau dort
> an — wir nutzen ein fertiges, bereits trainiertes Transformer-Modell **praktisch**, statt
> Attention erneut von Grund auf zu erklären.

Unser Running Example bleibt der Kfz-Versicherer — nur diesmal sind die Eingaben keine Tabellen
oder Fotos, sondern **Freitext**: Kundenbewertungen der App und Rückmeldungen zur
Schadensabwicklung.

## Schritt 1 — Tokenisierung von Hand nachvollziehen

Wir starten mit einem Beispielsatz aus dem Versicherer-Kontext:

> *"Die Schadensabwicklung war unfair und hat lange gedauert."*

Die einfachste denkbare Tokenisierung: den Satz einfach an jedem Leerzeichen aufteilen. Das ist
die **Word-Level-Tokenisierung** in ihrer naivsten Form — kein Modell, keine Bibliothek, nur
Python-Bordmittel.

In [ ]:
# I DO: Naive Tokenisierung per .split() - ein Wort pro Leerzeichen

satz = "Die Schadensabwicklung war unfair und hat lange gedauert."

woerter_naiv = satz.split()  # zerlegt den String ueberall dort, wo ein Leerzeichen steht

print(f"Anzahl 'Tokens': {len(woerter_naiv)}")
print(woerter_naiv)

### Schritt-für-Schritt-Breakdown

1. `.split()` zerlegt den Satz ausschließlich an **Leerzeichen** — das letzte Element `"gedauert."`
   enthält noch den Punkt, weil `.split()` keine Ahnung von Satzzeichen hat.
2. Das zusammengesetzte Wort `"Schadensabwicklung"` bleibt als **ein einziger Block** stehen —
   für ein Modell wäre das ein extrem seltenes, u. U. komplett unbekanntes Wort, wenn es nicht
   exakt in dieser Schreibweise im Trainingsvokabular vorkam.

> ⚠️ **Common Pitfall:**
> Naive Word-Level-Tokenisierung scheitert an genau den Wörtern, die in der deutschen Sprache
> besonders häufig vorkommen: **Komposita** (zusammengesetzte Wörter wie
> "Schadensabwicklung", "Kfz-Haftpflichtversicherung", "Parkplatzunfall"). Jedes neue Kompositum
> wäre ein komplett neues, nie gesehenes Vokabular-Element — das Modell könnte damit nichts
> anfangen.

Schauen wir uns jetzt an, wie ein echter, vortrainierter Tokenizer mit genau diesem Problem
umgeht.

> 💡 **Good to know:**
> Der erste Ausführungslauf der nächsten Zelle lädt den Tokenizer **und** das Sentiment-Modell
> von der HuggingFace-Plattform herunter — das kann beim ersten Mal **1–2 Minuten** dauern. Bei
> jedem weiteren Ausführen ist das Modell lokal zwischengespeichert und lädt sofort.

In [ ]:
# I DO: Echten Tokenizer UND das Sentiment-Modell laden
# Wir laden hier direkt beides zusammen, weil wir das Modell gleich in Schritt 2 (Embeddings)
# und Schritt 3 (Klassifikation) wiederverwenden - kein zweiter Download noetig.

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_id = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
model.eval()  # Evaluation-Modus: kein Training, nur Vorhersagen

print("Modell geladen. Klassen (id2label):", model.config.id2label)

> 💡 **Good to know:**
> `nlptown/bert-base-multilingual-uncased-sentiment` ist **mehrsprachig** (Deutsch, Englisch,
> Französisch, …) und wurde auf 629.000 Produktbewertungen trainiert. Der Output sind **5
> Klassen** — nicht "positiv/negativ/neutral", sondern Sternebewertungen von "1 star" bis
> "5 stars" (feinkörniges Sentiment, wie in der Vorlesung besprochen).

Jetzt tokenisieren wir denselben Satz noch einmal — diesmal mit dem echten Subword-Tokenizer des
Modells.

In [ ]:
# I DO: Subword-Tokenisierung mit dem echten HuggingFace-Tokenizer

tokens_subword = tokenizer.tokenize(satz)

print(f"Anzahl Subword-Tokens: {len(tokens_subword)}")
print(tokens_subword)

### Schritt-für-Schritt-Breakdown

1. `tokenizer.tokenize(satz)` zerlegt den Satz nicht an Leerzeichen, sondern anhand eines
   **trainierten Vokabulars** aus häufigen Wort- und Wortteil-Bausteinen (Subword-Tokenisierung,
   Prinzip: Byte Pair Encoding).
2. Achte auf `"schaden"` gefolgt von `"##sa"`, `"##b"`, `"##wicklung"` — das lange Kompositum
   `"Schadensabwicklung"` wurde in **vier** kleinere, im Training viel häufigere Bausteine
   zerlegt.
3. Das Präfix `##` bedeutet: "Dieses Stück wird **direkt** an das vorherige Token angehängt, es
   ist **kein** eigenständiges Wort." Ohne `##` würde `"wicklung"` wie ein eigenes Wort aussehen.

> ⚠️ **Common Pitfall:**
> `tokenizer.tokenize()` gibt Dir lesbare Text-Bausteine zurück — das Modell selbst "sieht" davon
> aber nur **Zahlen** (Token-IDs). `tokenizer.convert_ids_to_tokens()` macht diesen Schritt
> sichtbar, wenn Du von IDs zurück zu Text möchtest.

In [ ]:
# I DO: Von Token-IDs zurueck zu lesbaren Tokens (und umgekehrt)

token_ids = tokenizer.convert_tokens_to_ids(tokens_subword)
zurueck_zu_tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("Tokens:   ", tokens_subword)
print("Token-IDs:", token_ids)
print("Zurueck:  ", zurueck_zu_tokens)

> 💡 **Good to know:**
> `convert_tokens_to_ids()` und `convert_ids_to_tokens()` sind zueinander inverse Operationen —
> die Nachschlagetabelle (Vokabular) funktioniert in beide Richtungen. Genau diese Token-IDs sind
> es, die gleich in Schritt 2 durch das Embedding in Vektoren übersetzt werden.

Jetzt bist Du dran: Probiere die Subword-Tokenisierung an einem eigenen, zusammengesetzten Wort
aus dem Versicherer-Kontext aus.

> 🎯 **Your Task:**
> Ersetze `satz_neu` unten durch einen eigenen Satz mit einem langen deutschen Kompositum (z. B.
> "Kfz-Haftpflichtversicherung", "Reparaturkostenvoranschlag" oder ein eigenes Wort) und
> beobachte, in wie viele Subword-Bausteine der Tokenizer es zerlegt.

In [ ]:
# WE DO: Eigenen Satz mit einem langen Kompositum tokenisieren

satz_neu = "Der Reparaturkostenvoranschlag lag bei 8.500 EUR."  # <- hier eigenen Satz einsetzen

tokens_neu = tokenizer.tokenize(satz_neu)
print(f"Anzahl Subword-Tokens: {len(tokens_neu)}")
print(tokens_neu)

### Zwischenfazit

Du hast jetzt den Unterschied zwischen naiver Word-Level-Tokenisierung (`.split()`) und echter
Subword-Tokenisierung live gesehen: Der HuggingFace-Tokenizer zerlegt seltene, zusammengesetzte
Wörter automatisch in häufigere Bausteine. Als Nächstes schauen wir uns an, was mit diesen
Token-IDs eigentlich passiert, sobald sie in das Modell wandern: die **Embeddings**.

## Schritt 2 — Embeddings anschaulich

Ein **Embedding** übersetzt jede Token-ID in einen Vektor — eine Liste von Zahlen, die die
Bedeutung des Tokens im n-dimensionalen Raum codiert. Wörter mit ähnlicher Bedeutung sollen dabei
**nahe beieinander** liegen, unähnliche Wörter **weit auseinander**. Gemessen wird diese Nähe
meist über die **Kosinus-Ähnlichkeit**: Werte nahe **1** bedeuten "sehr ähnlich", Werte nahe
**0** "kaum verwandt", negative Werte "gegensätzlich".

Um das Prinzip greifbar zu machen, bauen wir zuerst ein **stark vereinfachtes** Spielzeug-Beispiel
von Hand: drei selbst gewählte Zahlen pro Wort statt der hunderten Dimensionen, die ein echtes
Modell nutzt.

In [ ]:
# I DO: Ein winziges, von Hand gebautes Embedding fuer vier Woerter

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Drei frei erfundene Dimensionen pro Wort, so gewaehlt, dass "gut" und "super" (beide positiv)
# nah beieinander liegen, "schlecht" in die entgegengesetzte Richtung zeigt, und "Auto" (ein
# neutrales Substantiv) weder mit positiv noch mit negativ viel zu tun hat.
embedding_spielzeug = {
    "gut":      np.array([0.90,  0.80, 0.10]),
    "super":    np.array([0.95,  0.75, 0.05]),
    "schlecht": np.array([-0.90, -0.80, 0.10]),
    "Auto":     np.array([0.10, -0.05, 0.90]),
}

for wort, vektor in embedding_spielzeug.items():
    print(f"{wort:>10}: {vektor}")

Jetzt berechnen wir die Kosinus-Ähnlichkeit zwischen allen Wortpaaren mit
`sklearn.metrics.pairwise.cosine_similarity`.

In [ ]:
# I DO: Kosinus-Aehnlichkeit zwischen allen Wortpaaren berechnen

woerter = list(embedding_spielzeug.keys())

for i in range(len(woerter)):
    for j in range(i + 1, len(woerter)):
        wort_a, wort_b = woerter[i], woerter[j]
        vektor_a = embedding_spielzeug[wort_a].reshape(1, -1)  # cosine_similarity erwartet 2D-Arrays
        vektor_b = embedding_spielzeug[wort_b].reshape(1, -1)
        aehnlichkeit = cosine_similarity(vektor_a, vektor_b)[0, 0]
        print(f"{wort_a:>10} <-> {wort_b:<10}: {aehnlichkeit:+.3f}")

### Schritt-für-Schritt-Breakdown

1. `"gut"` und `"super"` liegen mit **≈ +0,998** extrem nah beieinander — beide Wörter zeigen im
   Vektorraum fast in dieselbe Richtung, weil wir sie absichtlich ähnlich gewählt haben (genau wie
   semantisch ähnliche Wörter in einem echten, trainierten Embedding).
2. `"gut"` und `"schlecht"` liegen bei **≈ −0,987** — praktisch entgegengesetzt, wie zwei Pfeile,
   die in fast genau entgegengesetzte Richtungen zeigen.
3. `"gut"` und `"Auto"` liegen bei **≈ +0,128** — nahe 0, also **kaum verwandt**: ein neutrales
   Substantiv hat wenig mit einer positiven Bewertung zu tun.

> 💡 **Good to know:**
> Das ist genau dasselbe Prinzip wie das berühmte Beispiel aus der Vorlesung
> "König − Mann + Frau ≈ Königin": Richtungen und Abstände im Vektorraum codieren Bedeutung.
> "König" und "Königin" liegen dort nah beieinander (Ähnlichkeit ≈ 0,95), "König" und "Auto" weit
> auseinander (≈ 0,08) — exakt das Muster, das wir hier von Hand nachgebaut haben.

Jetzt experimentierst Du selbst mit einem eigenen Wort.

> 🎯 **Your Task:**
> Ergänze unten ein eigenes Wort mit einem eigenen 3er-Vektor zum Dictionary (z. B. `"schnell"`,
> ähnlich zu `"gut"`/`"super"` gewählt) und beobachte seine Ähnlichkeit zu den anderen Wörtern.

In [ ]:
# WE DO: Eigenes Wort zum Spielzeug-Embedding hinzufuegen

embedding_spielzeug["schnell"] = np.array([0.85, 0.70, 0.20])  # <- hier eigenen Vektor probieren

for wort in woerter:
    vektor_a = embedding_spielzeug["schnell"].reshape(1, -1)
    vektor_b = embedding_spielzeug[wort].reshape(1, -1)
    aehnlichkeit = cosine_similarity(vektor_a, vektor_b)[0, 0]
    print(f"schnell <-> {wort:<10}: {aehnlichkeit:+.3f}")

> ⚠️ **Common Pitfall — Realitätscheck:**
> Unser Spielzeug-Beispiel oben ist bewusst **von Hand konstruiert**, damit die Ähnlichkeiten
> "sauber" herauskommen. Echte Embeddings eines Transformer-Modells wie unserem geladenen BERT
> sehen in der Praxis **unordentlicher** aus — schauen wir uns das kurz ehrlich an, bevor wir
> weitermachen.

In [ ]:
# I DO: Echte Embedding-Schicht unseres geladenen Modells anschauen (Realitaetscheck)

embedding_schicht = model.get_input_embeddings()  # die Embedding-Matrix des BERT-Modells

echte_woerter = ["gut", "super", "schnell", "auto"]
echte_vektoren = {}

for wort in echte_woerter:
    token_id = tokenizer.convert_tokens_to_ids(wort)  # setzt voraus, dass 'wort' EIN Token ist
    echte_vektoren[wort] = embedding_schicht.weight[token_id].detach().numpy()

print(f"Dimensionen eines echten Embedding-Vektors: {echte_vektoren['gut'].shape}")

for i in range(len(echte_woerter)):
    for j in range(i + 1, len(echte_woerter)):
        a, b = echte_woerter[i], echte_woerter[j]
        aehnlichkeit = cosine_similarity(
            echte_vektoren[a].reshape(1, -1), echte_vektoren[b].reshape(1, -1)
        )[0, 0]
        print(f"{a:>8} <-> {b:<8}: {aehnlichkeit:+.3f}")

> 💡 **Good to know:**
> Ein echter Embedding-Vektor hat hier **768 Dimensionen** statt unserer 3 erfundenen — und die
> Ähnlichkeiten wirken auf den ersten Blick weniger "sauber" (z. B. `"super"` und `"auto"` teils
> ähnlicher als `"gut"` und `"super"`). Das liegt daran, dass diese rohe Embedding-Schicht nur der
> **erste Baustein** eines Transformers ist — die eigentliche, kontextabhängige Bedeutung
> entsteht erst durch die **Attention-Schichten darüber**, die wir hier nicht einzeln
> ansehen. Reine Word2Vec-Embeddings (wie im "König−Mann+Frau"-Beispiel) sind dagegen direkt für
> Wortähnlichkeit trainiert — deshalb wirken sie "sauberer" als die Rohschicht eines
> BERT-Klassifikationsmodells.

### Zwischenfazit

Du hast das Embedding-Prinzip jetzt an einem sauberen Spielzeug-Beispiel **und** an echten
Modell-Gewichten gesehen. Zeit, das komplette Modell für seine eigentliche Aufgabe zu nutzen:
Sentiment aus echten Kfz-Versicherer-Sätzen vorhersagen.

## Schritt 3 — Inferenz auf eigenen Kfz-Versicherer-Sätzen

Jetzt setzen wir die komplette Pipeline zusammen: Tokenisieren → Forward Pass (`torch.no_grad()`)
→ `torch.softmax()` → `argmax()` → `model.config.id2label`. Wir testen fünf neue Sätze mit
bewusst unterschiedlicher Tonalität — sehr positiv, sehr negativ, neutral, sowie zwei
"Grenzfälle" mit Sarkasmus bzw. Negation.

In [ ]:
# I DO: Fuenf neue Kfz-Versicherer-Beispielsaetze mit unterschiedlicher Tonalitaet

beispielsaetze = [
    "Die Kfz-Versicherung hat den Schaden super schnell reguliert, ich bin richtig begeistert!",
    "Absolute Frechheit, seit Wochen keine Rueckmeldung und die Hotline legt einfach auf.",
    "Der Schaden wurde wie im Vertrag beschrieben bearbeitet, nichts Besonderes.",
    "Toll, wieder drei Wochen auf die Rueckmeldung gewartet, genau das habe ich mir gewuenscht.",
    "Der Service war nicht schlecht, aber auch nicht wirklich gut.",
]

for text in beispielsaetze:
    # 1. Tokenisieren (truncation=True kappt zu lange Texte auf die Modell-Maximallaenge)
    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    # 2. Forward Pass ohne Gradientenberechnung (wir wollen nur vorhersagen, nicht trainieren)
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Logits -> Wahrscheinlichkeiten -> Klasse mit hoechster Wahrscheinlichkeit
    logits = outputs.logits
    wahrscheinlichkeiten = torch.softmax(logits, dim=1)
    predicted_id = logits.argmax(dim=1).item()
    confidence = wahrscheinlichkeiten[0, predicted_id].item()
    label = model.config.id2label[predicted_id]

    print(f"{label:>8} ({confidence:.3f})  <-  {text}")

### Schritt-für-Schritt-Breakdown

1. **Sehr positiv** ("...richtig begeistert!") → **5 stars (0,910)** — hohe Konfidenz, passt.
2. **Sehr negativ** ("Absolute Frechheit...") → **1 star (0,988)** — sehr hohe Konfidenz, passt.
3. **Neutral** ("...nichts Besonderes.") → **3 stars (0,551)** — richtige Richtung, aber mit
   **niedriger** Konfidenz: Das Modell ist sich hier deutlich unsicherer.
4. **Sarkasmus** ("Toll, wieder drei Wochen...") → **5 stars (0,773)** — und das ist **falsch**!
   Der Satz ist eindeutig negativ gemeint, das Modell erkennt aber nur das Wort "Toll" und den
   positiven Grundton, nicht die sarkastische Wendung.
5. **Negation** ("...nicht schlecht, aber auch nicht wirklich gut.") → **3 stars (0,705)** — eine
   vertretbare neutrale Einschätzung, aber die Konfidenz liegt nur knapp über der Grenze.

> ⚠️ **Common Pitfall:**
> Fall 4 zeigt exakt die Grenze von Sentimentanalyse, die schon in der Vorlesung angesprochen
> wurde: **Sarkasmus** wird oft falsch **und** mit hoher Scheinsicherheit vorhergesagt. Die
> Konfidenz allein schützt hier nicht zuverlässig vor Fehlern — sie ist ein nützliches, aber kein
> perfektes Signal.

> 💡 **Good to know:**
> Aus der Vorlesung: Konfidenzwerte **unter 0,7** gelten als Signal für **manuelle Prüfung** statt
> automatischer Verarbeitung. Fall 3 (0,551) läge klar in dieser Zone — Fall 5 (0,705) knapp
> darüber, ist aber auch kein Fall, bei dem man blind vertrauen sollte.

Jetzt bist Du dran: Verarbeite mehrere Sätze **gleichzeitig** in einem Batch statt einzeln
nacheinander.

> 🎯 **Your Task:**
> Führe die Zelle unten aus und beobachte, dass `padding=True` alle Sätze im Batch auf dieselbe
> Länge auffüllt (kürzere Sätze bekommen Auffüll-Tokens angehängt) — die Ergebnisse pro Satz
> bleiben dabei identisch zur Einzelverarbeitung oben.

In [ ]:
# WE DO: Mehrere Saetze gleichzeitig als Batch verarbeiten (padding=True)

batch_saetze = beispielsaetze[:3]  # die ersten drei Saetze von oben als Batch

# padding=True fuellt kuerzere Saetze mit Padding-Tokens auf, damit alle im Batch dieselbe
# Laenge haben - das braucht ein Tensor, der mehrere Saetze auf einmal speichern soll.
inputs_batch = tokenizer(batch_saetze, return_tensors="pt", truncation=True, padding=True)
print("Form des Input-Tensors (Batch, Token-Laenge):", tuple(inputs_batch["input_ids"].shape))

with torch.no_grad():
    outputs_batch = model(**inputs_batch)

wahrscheinlichkeiten_batch = torch.softmax(outputs_batch.logits, dim=1)
predicted_ids_batch = outputs_batch.logits.argmax(dim=1)

for i, text in enumerate(batch_saetze):
    predicted_id = predicted_ids_batch[i].item()
    confidence = wahrscheinlichkeiten_batch[i, predicted_id].item()
    label = model.config.id2label[predicted_id]
    print(f"{label:>8} ({confidence:.3f})  <-  {text}")

> 💡 **Good to know:**
> `inputs_batch["attention_mask"]` (kannst Du Dir mit `print(inputs_batch["attention_mask"])`
> anzeigen lassen) sagt dem Modell, welche Token echte Wörter sind und welche nur
> Auffüll-Padding — so beeinflussen die Padding-Tokens die Vorhersage nicht. Das ist der Grund,
> warum die Ergebnisse im Batch identisch zu den Einzelergebnissen bleiben, obwohl Sätze
> unterschiedlich lang sind.

### Zwischenfazit

Du hast jetzt die komplette Inferenz-Pipeline sowohl einzeln als auch im Batch angewendet — und
gesehen, wo sie zuverlässig funktioniert (klare Tonalität) und wo sie an ihre Grenzen stößt
(Sarkasmus). Jetzt verpackst Du diese Pipeline selbst in eine wiederverwendbare Funktion.

## YOU DO — Übung 1: `analysiere_sentiment(text)` vervollständigen

> 🎯 **Your Task:**
> Vervollständige die Funktion `analysiere_sentiment(text)` unten. Sie soll die komplette
> Pipeline aus Schritt 3 kapseln: Tokenisieren → Forward Pass → Softmax → Klasse mit höchster
> Wahrscheinlichkeit → Label + Konfidenz zurückgeben. Orientiere Dich exakt am Muster aus dem
> "I DO"-Beispiel weiter oben.

In [ ]:
# =========================================================================
# 🎯 EXERCISE: analysiere_sentiment(text) vervollstaendigen
# =========================================================================
# Instruction: Ersetze die ??? durch gueltigen Code (siehe Muster in Schritt 3 oben).

def analysiere_sentiment(text):
    # 1. Text tokenisieren
    inputs = ???

    # 2. Forward Pass ohne Gradientenberechnung
    with torch.no_grad():
        outputs = ???

    # 3. Logits -> Wahrscheinlichkeiten -> Klasse mit hoechster Wahrscheinlichkeit
    logits = outputs.logits
    wahrscheinlichkeiten = ???
    predicted_id = ???
    confidence = wahrscheinlichkeiten[0, predicted_id].item()
    label = ???

    return label, confidence


# Drei neue Testsaetze, die wir noch nicht verwendet haben
testsaetze = [
    "Ich bin von der schnellen Bearbeitung meines Schadensfalls total begeistert.",
    "Die Versicherung hat meine Anfrage komplett ignoriert, das ist inakzeptabel.",
    "Der Ablauf war okay, nichts Weltbewegendes, aber auch keine Probleme.",
]

for satz in testsaetze:
    label, confidence = analysiere_sentiment(satz)
    print(f"{label:>8} ({confidence:.3f})  <-  {satz}")

> 💡 **Good to know:**
> Diese Funktion ist jetzt Dein wiederverwendbarer Baustein — genau eine Funktion, die Du für
> jeden beliebigen Text aufrufen kannst, ohne die Pipeline jedes Mal neu auszuschreiben. Im
> nächsten Schritt nutzen wir sie (in einer Schleife) für eine ganze Liste von
> Kundenbewertungen.

## YOU DO — Übung 2: Batch-Verarbeitung mit `pandas`

Der Versicherer hat eine Liste von zehn neuen App-Bewertungen gesammelt (synthetisch, aber
realistisch). Wir wollen sie automatisch klassifizieren und die Ergebnisse übersichtlich in einer
Tabelle sammeln — genau die Art von Aufgabe, die in der Praxis hundertfach am Tag anfällt.

In [ ]:
# I DO: Zehn synthetische Kundenbewertungen (vorgegeben)

kundenbewertungen = [
    "Die App ist super uebersichtlich, ich finde alles sofort. Klasse gemacht!",
    "Nach dem Unfall hat mir mein Sachbearbeiter jeden Schritt genau erklaert. Vielen Dank!",
    "Die Wartezeit in der Hotline war eine Katastrophe, ueber 40 Minuten in der Warteschleife.",
    "Meine Praemie wurde ohne Ankuendigung erhoeht, das finde ich nicht in Ordnung.",
    "Der Vertrag wurde wie besprochen angepasst, alles im gruenen Bereich.",
    "Es hat funktioniert, war aber nichts Aussergewoehnliches.",
    "Ganz nett, aber die Website haengt manchmal ein bisschen.",
    "Endlich mal ein Versicherer, der schnell reagiert und fair reguliert!",
    "Der Kundenservice war unhoeflich und hat mein Anliegen einfach ignoriert.",
    "Klasse, jetzt muss ich zum dritten Mal dieselben Unterlagen einreichen, wirklich grossartig.",
]

print(f"Anzahl Bewertungen: {len(kundenbewertungen)}")

> 🎯 **Your Task:**
> Vervollständige die Schleife unten: Rufe für jede Bewertung `analysiere_sentiment()` auf und
> sammle Text, Label und Konfidenz in einer Liste von Dictionaries. Baue daraus einen
> `pandas.DataFrame`, sortiere ihn nach Konfidenz (aufsteigend) und markiere die **drei
> unsichersten Fälle mit Konfidenz unter 0,7** als `"manuell pruefen"`.

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Batch-Verarbeitung + DataFrame + Konfidenz-Flag
# =========================================================================
# Instruction: Ersetze die ??? durch gueltigen Code.

import pandas as pd

ergebnisse = []

for bewertung in kundenbewertungen:
    label, confidence = ???  # <- analysiere_sentiment() auf 'bewertung' anwenden
    ergebnisse.append({
        "text": bewertung,
        "label": ???,
        "confidence": ???,
    })

df_ergebnisse = pd.DataFrame(ergebnisse)

# Nach Konfidenz aufsteigend sortieren (unsicherste Faelle zuerst)
df_ergebnisse = df_ergebnisse.sort_values("???", ascending=True).reset_index(drop=True)

# Die 3 unsichersten Faelle markieren, aber nur wenn ihre Konfidenz wirklich unter 0.7 liegt
df_ergebnisse["pruefstatus"] = "automatisch verarbeitet"
unsicherste_3 = df_ergebnisse.index[:3]
df_ergebnisse.loc[unsicherste_3, "pruefstatus"] = df_ergebnisse.loc[unsicherste_3, "confidence"].apply(
    lambda c: "manuell pruefen" if c < 0.7 else "automatisch verarbeitet"
)

df_ergebnisse

### Schritt-für-Schritt-Breakdown

- Die drei unsichersten Bewertungen liegen alle deutlich **unter 0,7 Konfidenz** — genau die
  Fälle, die laut Vorlesungsfolie an einen Menschen zur manuellen Prüfung weitergeleitet werden
  sollten, statt sie automatisch als "erledigt" zu markieren.
- Schau Dir zusätzlich die Bewertung *"Klasse, jetzt muss ich zum dritten Mal dieselben
  Unterlagen einreichen, wirklich grossartig."* an: Sie wird mit **hoher** Konfidenz als "5 stars"
  eingestuft — obwohl der Satz eindeutig sarkastisch und negativ gemeint ist!

> ⚠️ **Common Pitfall:**
> Das zeigt: Eine **hohe** Konfidenz ist kein Garant für ein richtiges Ergebnis — sie zeigt nur,
> wie sicher sich das Modell **selbst** einschätzt. Bei Sarkasmus kann diese Selbsteinschätzung
> täuschend hoch ausfallen. Die Konfidenzschwelle von 0,7 fängt zuverlässig die **unsicheren**
> Fälle ab, aber nicht die **selbstsicher falschen**.

### Zwischenfazit

Du hast jetzt eine komplette, realistische Mini-Pipeline gebaut: zehn Kundenbewertungen
automatisch klassifiziert, nach Konfidenz sortiert und die unsichersten Fälle für eine manuelle
Prüfung markiert — genau der Workflow, den ein Versicherer im Kundenservice-Team einsetzen
würde.

## Mini-Exercise — Reflexionsfrage: Warum scheitert das Modell an Sarkasmus?

Wir haben mehrfach gesehen, dass sarkastische Sätze wie *"Toll, wieder drei Wochen auf die
Rückmeldung gewartet..."* falsch — und teils sogar mit hoher Konfidenz falsch — klassifiziert
werden.

> 🎯 **Your Task:**
> Beantworte die folgenden zwei Fragen als Kommentare bzw. `print()`-Ausgaben in der Zelle
> unten.
>
> **Frage 1:** Warum liefert das Modell bei einem sarkastischen Satz ein falsches oder unsicheres
> Ergebnis? Nenne **einen** konkreten Grund, der mit der Funktionsweise des Modells zu tun hat.
>
> **Frage 2:** Was würde helfen, um Sarkasmus zukünftig besser zu erkennen — mehr
> Trainingsdaten, mehr Kontext (z. B. vorherige Nachrichten desselben Kunden), oder Fine-Tuning
> auf versicherungsspezifische Sprache? Begründe kurz.

In [ ]:
# TODO: Trage Deine zwei Antworten als Kommentare bzw. print()-Ausgaben ein.

antwort_1 = "???"
antwort_2 = "???"

print(f"Antwort 1: {antwort_1}")
print(f"Antwort 2: {antwort_2}")

## Summary & Key Takeaways

- **Tokenisierung:** Naive Word-Level-Tokenisierung (`.split()`) scheitert an deutschen
  Komposita. Der echte HuggingFace-**Subword-Tokenizer** zerlegt seltene, zusammengesetzte Wörter
  wie "Schadensabwicklung" automatisch in häufigere Bausteine (`##`-Präfix = "direkt angehängt").
- **Embeddings** übersetzen Token-IDs in Vektoren; **Kosinus-Ähnlichkeit** misst semantische Nähe.
  Unser Spielzeug-Beispiel zeigte ein sauberes Muster (gut↔super ≈ +0,998, gut↔schlecht ≈ −0,987),
  echte BERT-Rohembeddings sehen unordentlicher aus — die eigentliche Bedeutung entsteht erst
  durch Attention.
- **Praktische Pipeline:** `AutoTokenizer.from_pretrained()` + `AutoModelForSequenceClassification.from_pretrained()`
  laden ein fertig trainiertes Modell; Tokenisieren → `torch.no_grad()`-Forward-Pass →
  `torch.softmax()` → `argmax()` → `model.config.id2label` liefert Label + Konfidenz.
- **Batch-Verarbeitung** mit `padding=True` verarbeitet mehrere Sätze gleichzeitig — der
  `attention_mask` sorgt dafür, dass Auffüll-Tokens die Vorhersage nicht verfälschen.
- **Konfidenz als Signal:** Werte unter 0,7 markieren Fälle für die manuelle Prüfung — aber eine
  **hohe** Konfidenz schützt nicht zuverlässig vor Fehlern, wie das sarkastische Beispiel mit
  0,932 Konfidenz gezeigt hat.
- **Ehrliche Grenze:** Sarkasmus und Negation bleiben Schwachstellen von Sentimentanalyse-Modellen
  — Fine-Tuning mit eigenen, versicherungsspezifischen Beispielen (100–500 reichen laut
  Vorlesung) kann diese Lücke gezielt verkleinern.